# Notebook to demo point near point analysis using `JOIN_QR`.

In [4]:
import geofunctions as S
import geopandas as gp
import pyspark.sql.functions as F
from pyspark.sql import Window

### Create left hand side (LHS) points.

NOTE: THIS HAS TO BE EXECUTED IN A PRO NOTEBOOK WITH MyPoints IN THE TOC.

In [ ]:
my_points = S.to_spark("MyPoints")

In [ ]:
lhs = my_points.select(
    F.col("OBJECTID").alias("lid"),
    F.col("Shape").alias("geom"),
)

### Create right hand side (RHS) points.

In [ ]:
rhs = my_points.select(
    F.col("OBJECTID").alias("rid"),
    F.col("Shape").alias("geom"),
)

### Define search distance and QR cell size.

Here you can tune based on use case the cell size in proportion to the dist.

In [ ]:
dist = 2000.0
cell = dist * 5.0

### Define window to find nearest row.

In [ ]:
over = Window.partitionBy("lid").orderBy("dist")

### Perform spatial join.

In [ ]:
res = (
    lhs
    # Here we pass both cell and dist as we are joining point to point.
    .join_qr(rhs, cell, dist)
    # Make sure we are not self-joining :-)
    .filter("lid != rid")
    # Calculate the distance between the LHS and RHS point.
    .withColumn("dist", S.st_distance())
    # Rank by distance all the RHS within distance.
    .filter(F.col("dist") < dist)
    .withColumn("rn", F.row_number().over(over))
    # Get the smallest (first one)
    .filter("rn==1")
    .drop("rn")
    # Create a line between the two points.
    .select(
        "lid",
        "rid",
        S.st_polyline("lgeom", "rgeom"),
        "dist",
    )
)

### Let's plot the results.

What is the nearest green point (rhs) to a red point (lhs) within search distance (yellow) ?

In [ ]:
res.to_feature_class("Nearest")